# GPT-generated summaries of category contents

In [1]:
import pandas as pd
from discovery_child_development import PROJECT_DIR, S3_BUCKET
import json

ENRICHED_DATA_DIR = PROJECT_DIR / 'outputs/enrichments'

In [2]:
OUTPUTS_DIR = ENRICHED_DATA_DIR / 'themes'
OUTPUTS_DIR.mkdir(exist_ok=True, parents=True)

In [3]:
from discovery_child_development.utils.openai_utils import client

In [4]:
# Path to topic information
PATH_TO_TOPICS = PROJECT_DIR / "discovery_child_development/pipeline/labelling/taxonomy_cat/prompts/topics.json"

# Load topic information
topics_dict = json.load(open(PATH_TO_TOPICS, 'r'))
topics = list(topics_dict.keys())
print(len(topics))

topics_df = []
for topic in topics_dict:
    topics_df.append({
        'topic': topic,
        'type': topics_dict[topic]['type'],
        'name': topics_dict[topic]['name'],
        # 'description': topics_dict[topic]['description']
    })
topics_df = (
    pd.DataFrame(topics_df)
    .sort_values(['type', 'topic',])
    .reset_index(drop=True)
    .replace('Family and home', 'Parenting')
    .rename(columns={'type': 'Type'})
)

38


# OpenAlex themes

In [9]:
# openalex_df = pd.read_csv(PROJECT_DIR / "outputs/data/tables/openalex_final.csv")

In [5]:
relevant_df = pd.read_csv(ENRICHED_DATA_DIR / 'relevant_labelled_df.csv')

In [14]:
len(relevant_df.query("Dataset == 'Publications'"))

38271

In [6]:
relevant_df.sample(2)

,id,text,Dataset,topics,Detection,Detection_,Management,Management_,year,country_code
31090,W3039283089,Vad berättas om arbete med jämställdhet inom d...,Publications,"inequality, preschool, inclusion",0.019161,0,0.858211,1,2020,NaN
23495,W2792343738,"Age correction in cognitive, linguistic, and m...",Publications,"neuroscience, infancy, cognitive, prenatal, ph...",0.999167,1,0.000072,0,2018,NaN


In [7]:
topics_df.head()

,topic,Type,name
0,genetics,Biosciences,Genetics
1,neuroscience,Biosciences,Neuroscience
2,operations,Child care & preschool,Operations
3,preschool,Child care & preschool,Preschool
4,arts,Development & learning,Expressive arts and design


In [16]:
from discovery_child_development.getters.openalex import get_sentence_embeddings

# Path to sentence embeddings
VECTORS_PATH = "data/outputs/vectors/"
VECTORS_FILE = "sentence_vectors_384_labelled.parquet"

In [17]:
# Load dataset sentence embeddings (all-MiniLM-L6-v2)
embeddings_all = (
    get_sentence_embeddings(
        s3_bucket=S3_BUCKET,
        filepath=VECTORS_PATH,
        filename=VECTORS_FILE,
        id="id",
    )
    .reset_index()
    # Simplify the id by removing https
    .assign(id=lambda df: df["id"].apply(lambda x: x.split("/")[-1]))
    .set_index("id")
)

In [18]:
len(embeddings_all)

51234

## Pre-process data

In [10]:
def get_overlaps(df: pd.DataFrame, categories: list, category_column: str) -> pd.DataFrame:
    """Fetch the datapoints that have all the categories in the list

    Args:
        df (pd.DataFrame): DataFrame containing the datapoints
        categories (list): List of categories
        category_column (str): Column name that contains the categories

    Returns:
        pd.DataFrame: DataFrame containing the datapoints that have all the categories in the list
    """
    return (
        df
        .copy()
        # transform comma separated string to list, account for nulls
        .assign(**{category_column: lambda x: x[category_column].fillna('').str.split(', ')})
        # filter the rows that have all the categories in the list
        .loc[lambda x: x[category_column].apply(lambda y: set(categories).issubset(y))]
    )

In [11]:
get_overlaps(relevant_df, ['ai2', 'income'], 'topics')

,id,text,Dataset,topics,Detection,Detection_,Management,Management_,year,country_code
18048,W2413786735,Early Childhood Developmental Status in Low- a...,Publications,"[health, nutrition, ai2, inequality, income, n...",9.991111e-01,1,0.000952,0,2016,NaN
18734,W2492801936,Early child development programmes: further ev...,Publications,"[mental_health, health, policy, neuroscience, ...",2.850358e-02,0,0.999995,1,2016,NaN
33897,W3158614866,Converging disciplines for assessing child dev...,Publications,"[mental_health, neuroscience, ai2, inclusion, ...",1.741255e-02,0,0.999996,1,2021,NaN
35221,W4310568231,"Concentrated poverty, ambient air pollution, a...",Publications,"[ai2, inequality, income, cognitive, infancy]",9.847409e-01,1,0.060228,0,2022,NaN
36012,W4289524760,Predictors of Early Childhood Development. Inc...,Publications,"[health, ai2, inclusion, income, emotional]",3.634219e-01,0,0.844976,1,2022,NaN
46714,W3090650878,PROTOCOL: Use of community participation inter...,Publications,"[health, rct, ai2, income, inequality, infancy]",2.061154e-09,0,1.000000,1,2020,NaN


In [135]:
themes = []

In [182]:
types = ["Innovation", "Area of learning", "Development", "Social", "General"]
topic1 = 'income'
topics2 = list(set(topics_df.query("Type == @types").topic.to_list()).difference([topic1]))

# for topic2 in topics2[0:1]: 
for topic2 in topics2: 

    for dataset in ["Publications", "Patents"]:

        # Get the documents that have both topics
        df = (
            get_overlaps(relevant_df, [topic1, topic2], 'topics')[['id', 'text', 'topics', 'Dataset']]
            .query("Dataset == @dataset")
            .assign(text_ = lambda x: 'ID: ' + x['id'] + ' | TEXT: ' + x['text'])
        )
        # Take 50 documents which might be most focussed on the two topics
        df = (
            df
            # shuffle
            .sample(len(df))
            # sort by number of topics
            .assign(n_topics = lambda df: df['topics'].apply(lambda x: len(x)))
            .sort_values('n_topics')
            # get the first 50
            .head(50)
        )

        topic1_name = topics_df.query("topic == @topic1").iloc[0]["name"]
        topic2_name = topics_df.query("topic == @topic2").iloc[0]["name"]

        topic1_type = topics_df.query("topic == @topic1").iloc[0]["Type"]
        topic2_type = topics_df.query("topic == @topic2").iloc[0]["Type"]

        abstract_texts = "\n\n".join(df.text_.to_list())

        gpt_message = f"You are an expert researcher on early childhood development. \
            Here are documents related to topics: Topic 1: {topic1_name} ({topic1_type}) and Topic 2: {topic2_name} ({topic2_type}). \
            Detect one to three distinct themes across these documents, that are closely related to the interaction between these two topics, and summarise these themes. \
            For each theme, write up to two sentences of summary, highlighting how the two topics overlap \
            and also two or three most interesting examples of innovations, technologies or interventions, referencing the alphanumeric ID of the document describing the example. \
            Focus on themes and examples that consider both topics together. \
            Here's an example: \n\n##Example\n\nTheme: [Theme name]\nSummary:[Summary of the theme]\nExample 1: \
            [Example of innovation, technology or intervention] (ID: [ID of the document] \nExample 2: [Example of innovation, technology or intervention] \
            (ID: [ID of the document])\n\n##Document texts\n\n {abstract_texts} \n\n##Themes\n\n"
        
        # Generate cluster descriptions
        print(f"Generating cluster themes for {topic1_name} and {topic2_name}")
        print(f"Dataset: {dataset}, number of documents: {len(df)}")
        messages = [
            {
                "role": "user",
                "content": gpt_message,
            }
        ]
        if len(df) > 0:
            chatgpt_output = client.chat.completions.create(
                model="gpt-4-turbo-2024-04-09",
                messages=messages,
                temperature=0.6,
                max_tokens=2000,
            )
            cluster_themes = chatgpt_output.choices[0].message.content
        else:
            print("No documents found for the given topics") 

        themes.append(
            {
                "topic1": topic1,
                "topic2": topic2,
                "dataset": dataset,
                "numb_docs": len(df),
                "cluster_themes": cluster_themes,
            }
        )
    

Generating cluster themes for Income and Labour market
Dataset: Publications, number of documents: 50
2024-04-26 17:01:28,964 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Generating cluster themes for Income and Labour market
Dataset: Patents, number of documents: 0
No documents found for the given topics
Generating cluster themes for Income and RCTs
Dataset: Publications, number of documents: 50
2024-04-26 17:01:56,700 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Generating cluster themes for Income and RCTs
Dataset: Patents, number of documents: 0
No documents found for the given topics
Generating cluster themes for Income and Non-tech assessments
Dataset: Publications, number of documents: 50
2024-04-26 17:02:20,313 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Generating cluster themes for Income and Non-tech assessments
Dataset:

In [174]:
df

,id,text,topics,Dataset,text_,n_topics


In [183]:
themes_df = pd.DataFrame(themes).rename(columns={'numb_docs': 'n_docs'})
# if numb_docs = 0, then make cluster_themes empty
themes_df.loc[themes_df['n_docs'] == 0, 'cluster_themes'] = ''

In [184]:
topic1

'income'

In [181]:
# Save json and save csv
themes_df.to_csv(OUTPUTS_DIR / f'topic_themes_{topic1}.csv', index=False)
themes_df.to_json(OUTPUTS_DIR / f'topic_themes_{topic1}.json', orient='records')

In [171]:
topics_df.to_csv(OUTPUTS_DIR / 'topics.csv', index=False)

In [118]:


# # Get cluster centroid indices
# centroids = get_cluster_centroids(cluster_df, embeddings)
# most_central = []
# for i in range(len(centroids)):
#     most_central.append(
#         get_n_most_central_vectors(embeddings, centroids[i], n=n_central)
#     )



Generating cluster themes for Data science and AI and Mathematics
Dataset: Publications, number of documents: 29
2024-04-23 12:24:09,124 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


## Summarisation